# L901 地方競馬 オッズ・出走表・結果 常時取得スクリプト

NAR (地方競馬) の全レースを対象に、常時起動で以下を取得・保存する。

## 取得タイミング
| タイミング | 取得内容 |
|---|---|
| 当日初回 (各レース) | 出走表 (馬名/騎手/調教師/オッズ) |
| 発走 T-30分 | 単勝・複勝・馬連 オッズスナップショット |
| 発走 T-10分 | 同上 |
| 発走 T-3分  | 同上 |
| 発走後 (確認できしだい) | レース結果 + 確定オッズ + 払戻 |

## ディレクトリ構成
```
C:/Users/ppny9/workspace/nar/
├── notebooks/#L901_odds_snapshot_v01.ipynb  ← 本ノートブック
├── data/
│   ├── schedule/{YYYYMMDD}.csv         当日スケジュール cache
│   ├── shutsuba/{race_id}.csv          出走表
│   ├── odds_snapshots/{race_id}_{T30|T10|T3}_{YYYYMMDD-HHMM}.csv
│   └── results/{race_id}_result.csv    確定オッズ + 払戻
└── logs/
```

## 注意
- Selenium WebDriver (Chrome headless) を使用 (NAR netkeiba odds は JS レンダリング)
- 単一 driver インスタンスを再利用 (起動コスト削減)
- 1リクエスト 3 秒間隔 (rate-limit 対策)


In [1]:
import os
import re
import sys
import json
import time
import traceback
from datetime import datetime, timedelta
from pathlib import Path
from time import sleep

import pandas as pd
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    NoSuchElementException, TimeoutException, WebDriverException
)


In [2]:
# ════════════════════════════════════════════════════════════
# 設定
# ════════════════════════════════════════════════════════════
RACE_DATE = datetime.now().strftime('%Y%m%d')   # 当日自動
# RACE_DATE = '20260618'   # 手動指定 (デバッグ用)

# スナップショット取得タイミング (発走前 分, ± 許容窓 分)
# T-60 広め (取りこぼし防止) / T-3,T-1 狭め (直前精度)
SNAPSHOT_CONFIG = [
    (60, 2.0),   # T-60: 基準点 (朝〜公開直後)
    (30, 1.0),   # T-30: 中期トレンド
    (15, 0.7),   # T-15: ミドルレンジ
    (10, 0.7),   # T-10: スマートマネー流入帯入り口
    ( 5, 0.5),   # T-5 : 本格スマートマネー
    ( 3, 0.3),   # T-3 : 最終駆け込み前半
    ( 1, 0.3),   # T-1 : 最終駆け込み (締切直前)
]
SNAPSHOT_OFFSETS_MIN = [c[0] for c in SNAPSHOT_CONFIG]   # 互換用

# レース結果取得タイミング
RESULT_AFTER_MIN = 5         # 発走後 何分後から結果取得を試みるか
RESULT_RETRY_MIN = 5         # 結果取得失敗時のリトライ間隔

# ループ間隔
CHECK_INTERVAL_SEC = 30                # スケジュール確認間隔
REQUEST_DELAY_SEC = 3                   # 連続リクエスト間隔 (Selenium共通)

# パス
NAR_ROOT = Path(r'C:/Users/ppny9/workspace/nar')
DATA_DIR = NAR_ROOT / 'data'
SCHEDULE_DIR = DATA_DIR / 'schedule'
SHUTSUBA_DIR = DATA_DIR / 'shutsuba'
SNAPSHOT_DIR = DATA_DIR / 'odds_snapshots'
RESULT_DIR   = DATA_DIR / 'results'
LOG_DIR      = NAR_ROOT / 'logs'
for d in [SCHEDULE_DIR, SHUTSUBA_DIR, SNAPSHOT_DIR, RESULT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Chrome オプション
def _new_chrome_options():
    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--window-size=1280,1024')
    opts.add_argument('--log-level=3')
    opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                       'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36')
    return opts

HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'),
    'Accept-Language': 'ja-JP,ja;q=0.9',
}

print(f'対象日       : {RACE_DATE}')
print(f'取得タイミング: {[f"T-{o}分(±{w}分)" for o, w in SNAPSHOT_CONFIG]}')
print(f'保存ルート   : {DATA_DIR}')


対象日       : 20260618
取得タイミング: ['T-60分(±2.0分)', 'T-30分(±1.0分)', 'T-15分(±0.7分)', 'T-10分(±0.7分)', 'T-5分(±0.5分)', 'T-3分(±0.3分)', 'T-1分(±0.3分)']
保存ルート   : C:\Users\ppny9\workspace\nar\data


In [3]:
# ════════════════════════════════════════════════════════════
# Selenium driver - 単一インスタンス管理
# ════════════════════════════════════════════════════════════
_driver = None

def get_driver():
    """driver を取得 (なければ作る)。クラッシュ時は再作成"""
    global _driver
    if _driver is None:
        _driver = webdriver.Chrome(options=_new_chrome_options())
        _driver.set_page_load_timeout(30)
        print(f'[INFO] Chrome driver 起動')
    return _driver

def reset_driver():
    """driver を再起動 (例外復旧用)"""
    global _driver
    try:
        if _driver:
            _driver.quit()
    except Exception:
        pass
    _driver = None
    return get_driver()

def safe_get(url, retries=2):
    """GET with retry on WebDriverException"""
    drv = get_driver()
    for attempt in range(retries + 1):
        try:
            drv.get(url)
            return drv
        except (TimeoutException, WebDriverException) as e:
            print(f'  [WARN] get失敗 (試行{attempt+1}): {e}')
            drv = reset_driver()
            time.sleep(REQUEST_DELAY_SEC)
    raise RuntimeError(f'safe_get 失敗: {url}')


In [4]:
# ════════════════════════════════════════════════════════════
# 当日 NAR スケジュール取得 (BS4 で静的 HTML 解析)
# ════════════════════════════════════════════════════════════
def get_nar_schedule(race_date: str) -> pd.DataFrame:
    """NAR の当日全レース一覧を取得。
    Returns DataFrame: race_id (12桁), start_time (HH:MM), venue
    """
    url = f'https://nar.netkeiba.com/top/race_list_sub.html?kaisai_date={race_date}'
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f'[WARN] NAR スケジュール取得失敗: {e}')
        return pd.DataFrame(columns=['race_id', 'start_time', 'venue'])

    soup = BeautifulSoup(resp.text, 'html.parser')
    races = []
    seen = set()

    # 各 race_id への href を探索
    for a in soup.find_all('a', href=True):
        m = re.search(r'race_id=(\d{12})', a['href'])
        if not m:
            continue
        race_id = m.group(1)
        if race_id[:4] != race_date[:4]:
            continue
        if race_id in seen:
            continue
        seen.add(race_id)

        # 発走時刻 — 親要素を辿って HH:MM を探す
        start_time = ''
        parent = a.parent
        for _ in range(6):
            if parent is None: break
            m2 = re.search(r'(\d{1,2}:\d{2})', parent.get_text())
            if m2:
                start_time = m2.group(1); break
            parent = parent.parent

        # 場所コード (race_id の 5-6文字目) と venue 名マッピング
        venue_code = race_id[4:6]
        venue_name = NAR_VENUE_NAME.get(venue_code, venue_code)
        races.append({'race_id': race_id, 'start_time': start_time,
                       'venue_code': venue_code, 'venue': venue_name})

    df = pd.DataFrame(races).drop_duplicates('race_id').reset_index(drop=True)
    # キャッシュ
    cache = SCHEDULE_DIR / f'{race_date}.csv'
    df.to_csv(cache, index=False, encoding='utf-8-sig')
    return df


# NAR 場所コード (netkeiba race_id 5-6 桁目) → 競馬場名
NAR_VENUE_NAME = {
    '30': '門別',
    '35': '盛岡',
    '36': '水沢',
    '42': '浦和',
    '43': '船橋',
    '44': '大井',
    '45': '川崎',
    '46': '金沢',
    '47': '笠松',
    '48': '名古屋',
    '50': '園田',
    '51': '姫路',
    '54': '高知',
    '55': '佐賀',
    '65': '帯広(ばんえい)',
}


In [5]:
# ════════════════════════════════════════════════════════════
# 出走表 (shutsuba) 取得 — Selenium
# ════════════════════════════════════════════════════════════
def fetch_shutsuba(race_id: str) -> pd.DataFrame:
    """出走表を取得。Returns DataFrame: 馬番, 馬名, 騎手, 調教師, race_name, venue, race_num"""
    url = f'https://nar.netkeiba.com/race/shutuba.html?race_id={race_id}'
    drv = safe_get(url)
    time.sleep(REQUEST_DELAY_SEC)

    race_name = ''; venue = ''; race_num = ''
    try:
        race_num = drv.find_element(By.CLASS_NAME, 'Race_Num').text.strip()
    except NoSuchElementException:
        pass
    try:
        rn = drv.find_element(By.CLASS_NAME, 'RaceName')
        race_name = rn.get_attribute('textContent').strip().split('\n')[0]
    except NoSuchElementException:
        pass
    try:
        spans = drv.find_elements(By.CSS_SELECTOR, '.RaceData02 span')
        if len(spans) >= 2:
            venue = spans[1].text.strip()
    except Exception:
        pass

    rows = drv.find_elements(By.CSS_SELECTOR, 'table.ShutubaTable > tbody > tr')
    entries = []
    for i, row in enumerate(rows, start=1):
        try:
            # 馬番: class="UmabanX" (X は馬番数字) → [class*="Umaban"] で部分一致
            uma_no = ''
            try:
                uma_no = row.find_element(By.CSS_SELECTOR, 'td[class*="Umaban"]').text.strip()
            except NoSuchElementException:
                cells = row.find_elements(By.TAG_NAME, 'td')
                uma_no = cells[1].text.strip() if len(cells) >= 2 else str(i)

            horse_name = row.find_element(By.CLASS_NAME, 'HorseInfo').text.strip()
            jockey = row.find_element(By.CLASS_NAME, 'Jockey').text.strip()
            try:
                trainer = row.find_element(By.CLASS_NAME, 'Trainer').text.strip()
            except NoSuchElementException:
                trainer = ''
            # 単勝オッズ: class="Popular Txt_R"
            try:
                odds = row.find_element(By.CSS_SELECTOR, 'td.Popular.Txt_R').text.strip()
            except NoSuchElementException:
                odds = ''
            entries.append([uma_no, horse_name, jockey, trainer, odds])
        except Exception as e:
            print(f'  [WARN] 行スキップ ({race_id} 行{i}): {e}')

    df = pd.DataFrame(entries, columns=['馬番', '馬名', '騎手', '調教師', '単勝オッズ'])
    df['race_id'] = race_id
    df['race_name'] = race_name
    df['venue'] = venue
    df['race_num'] = race_num
    df['fetched_at'] = datetime.now().isoformat(timespec='seconds')
    return df


def save_shutsuba(race_id: str, df: pd.DataFrame) -> Path:
    out = SHUTSUBA_DIR / f'{race_id}_shutsuba.csv'
    df.to_csv(out, index=False, encoding='utf-8-sig')
    print(f'  [SAVE shutsuba] {out.name} ({len(df)}頭)')
    return out

In [ ]:
# ════════════════════════════════════════════════════════════
# オッズ取得 — type=b1 (単勝/複勝)、b4 (馬連)
# ════════════════════════════════════════════════════════════

# NAR netkeiba オッズページで試みるセレクタ (優先順)
_TANFUKU_SELECTORS = [
    'table.RaceOdds_HorseList_Table > tbody > tr',
    'table.Odds_Table > tbody > tr',
    'table.Tansho > tbody > tr',
    '#odds-data-table tbody tr',
    'table[summary*="単勝"] tbody tr',
]

def fetch_odds_tan_fuku(race_id: str) -> pd.DataFrame:
    """単勝・複勝オッズを取得"""
    url = f'https://nar.netkeiba.com/odds/index.html?type=b1&race_id={race_id}'
    drv = safe_get(url)
    time.sleep(REQUEST_DELAY_SEC)

    # セレクタを順番に試みる
    rows = []
    matched_sel = None
    for sel in _TANFUKU_SELECTORS:
        rows = drv.find_elements(By.CSS_SELECTOR, sel)
        if rows:
            matched_sel = sel
            break

    if not rows:
        # デバッグ: ページ上の全テーブルを列挙
        tables = drv.find_elements(By.TAG_NAME, 'table')
        print(f'  [WARN] {race_id} 単複オッズテーブル未検出。'
              f'ページ上のtable数={len(tables)}')
        for t in tables[:8]:
            cls = t.get_attribute('class') or ''
            n = len(t.find_elements(By.TAG_NAME, 'tr'))
            print(f'    class="{cls[:60]}" rows={n}')
        return pd.DataFrame()

    print(f'  [DEBUG] tanfuku selector="{matched_sel}" ({len(rows)}行)')

    entries = []
    for row in rows:
        try:
            cells = row.find_elements(By.TAG_NAME, 'td')
            if len(cells) < 3:
                continue
            uma_no = None
            for c in cells:
                txt = c.text.strip()
                if re.fullmatch(r'\d{1,2}', txt) and uma_no is None:
                    uma_no = int(txt); break
            if uma_no is None:
                continue
            tan = None; fk_lo = None; fk_hi = None
            for c in cells:
                txt = c.text.strip()
                m_range = re.fullmatch(r'(\d+(?:\.\d+)?)\s*[-~–]\s*(\d+(?:\.\d+)?)', txt)
                m_single = re.fullmatch(r'\d+(?:\.\d+)?', txt)
                if m_range and fk_lo is None:
                    fk_lo = float(m_range.group(1)); fk_hi = float(m_range.group(2))
                elif m_single and tan is None and txt not in ('', '0'):
                    try:
                        tan = float(txt)
                    except ValueError:
                        pass
            entries.append({'umaban': uma_no, 'odds_tan': tan,
                             'odds_fukusho_lo': fk_lo, 'odds_fukusho_hi': fk_hi})
        except Exception:
            pass

    df = pd.DataFrame(entries).drop_duplicates('umaban').sort_values('umaban').reset_index(drop=True)
    if not df.empty and 'odds_tan' in df.columns:
        df['pop_tan'] = df['odds_tan'].rank(method='min').astype('Int64')
    return df


_UMAREN_SELECTORS = [
    'table.Odds_Table',
    'table.RaceOdds_HorseList_Table',
    'table.Umaren',
]

def fetch_odds_umaren(race_id: str) -> pd.DataFrame:
    """馬連オッズを取得
    NAR netkeiba の馬連テーブルは <th>馬番</th><td>オッズ</td> 構造。
    テーブルヘッダの <th> から基準馬番を取得し、各行の <th> と組み合わせてペアを生成する。
    """
    url = f'https://nar.netkeiba.com/odds/index.html?type=b4&race_id={race_id}'
    drv = safe_get(url)
    time.sleep(REQUEST_DELAY_SEC)

    tables = []
    for sel in _UMAREN_SELECTORS:
        tables = drv.find_elements(By.CSS_SELECTOR, sel)
        if tables:
            print(f'  [DEBUG] umaren selector="{sel}" ({len(tables)}テーブル)')
            break

    if not tables:
        all_tables = drv.find_elements(By.TAG_NAME, 'table')
        print(f'  [WARN] {race_id} 馬連テーブル未検出。table数={len(all_tables)}')
        for t in all_tables[:8]:
            cls = t.get_attribute('class') or ''
            n = len(t.find_elements(By.TAG_NAME, 'tr'))
            print(f'    class="{cls[:60]}" rows={n}')
        return pd.DataFrame(columns=['P1', 'P2', 'odds_umaren'])

    rows_out = []
    for tbl in tables:
        all_trs = tbl.find_elements(By.TAG_NAME, 'tr')

        # ── アプローチ1: td[0] が "N-M" 形式のペアテキスト ──────────────
        for tr in all_trs:
            tds = tr.find_elements(By.TAG_NAME, 'td')
            if len(tds) < 2:
                continue
            pair_txt = tds[0].text.strip()
            odds_txt = tds[1].text.strip()
            m = re.match(r'(\d+)\s*[-‐ー]\s*(\d+)', pair_txt)
            if not m:
                continue
            p1, p2 = int(m.group(1)), int(m.group(2))
            if p1 > p2:
                p1, p2 = p2, p1
            try:
                rows_out.append({'P1': p1, 'P2': p2, 'odds_umaren': float(odds_txt)})
            except ValueError:
                pass

        if rows_out:
            break  # アプローチ1 で取得できたら終了

        # ── アプローチ2: <th>基準馬番</th> + 各行 <th>相手馬番</th><td>オッズ</td> ──
        # テーブルヘッダ (最初の <th> が数字) から基準馬番を取得
        base_uma = None
        for tr in all_trs:
            ths = tr.find_elements(By.TAG_NAME, 'th')
            for th in ths:
                m = re.fullmatch(r'(\d{1,2})', th.text.strip())
                if m:
                    base_uma = int(m.group(1))
                    break
            if base_uma is not None:
                break

        if base_uma is None:
            continue

        for tr in all_trs:
            row_ths = tr.find_elements(By.TAG_NAME, 'th')
            row_tds = tr.find_elements(By.TAG_NAME, 'td')
            if not row_ths or not row_tds:
                continue
            # 行内 <th> が相手馬番
            m = re.fullmatch(r'(\d{1,2})', row_ths[-1].text.strip())
            if not m:
                continue
            p_other = int(m.group(1))
            if p_other == base_uma:
                continue
            p1, p2 = min(base_uma, p_other), max(base_uma, p_other)
            # オッズは <td> の中から数値を探す
            for td in row_tds:
                try:
                    odds = float(td.text.strip())
                    rows_out.append({'P1': p1, 'P2': p2, 'odds_umaren': odds})
                    break
                except ValueError:
                    pass

    if not rows_out:
        print(f'  [WARN] {race_id} 馬連オッズ行ゼロ。テーブル構造デバッグ:')
        for tbl in tables[:2]:
            for tr in tbl.find_elements(By.TAG_NAME, 'tr')[:6]:
                ths_txt = [th.text.strip() for th in tr.find_elements(By.TAG_NAME, 'th')]
                tds_txt = [td.text.strip() for td in tr.find_elements(By.TAG_NAME, 'td')]
                print(f'    th={ths_txt} td={tds_txt}')
        return pd.DataFrame(columns=['P1', 'P2', 'odds_umaren'])

    df = (pd.DataFrame(rows_out)
            .drop_duplicates(['P1', 'P2'])
            .sort_values(['P1', 'P2'])
            .reset_index(drop=True))
    return df


def save_odds_snapshot(race_id: str, label: str,
                        df_tf: pd.DataFrame, df_umaren: pd.DataFrame) -> tuple:
    """単勝/複勝 と 馬連 を別 CSV で保存"""
    ts_iso = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
    ts_fn  = datetime.now().strftime('%Y%m%d-%H%M')
    saved = []
    for kind, df in [('tanfuku', df_tf), ('umaren', df_umaren)]:
        if df is None or df.empty:
            print(f'  [WARN] {race_id} {label} {kind}: 空, スキップ')
            continue
        df = df.copy()
        df.insert(0, 'race_id', race_id)
        df.insert(1, 'snapshot_time', ts_iso)
        df.insert(2, 'label', label)
        df.insert(3, 'kind', kind)
        out = SNAPSHOT_DIR / f'{race_id}_{label}_{kind}_{ts_fn}.csv'
        df.to_csv(out, index=False, encoding='utf-8-sig')
        saved.append(out)
        print(f'  [SAVE odds] {out.name} ({len(df)}行)')
    return saved

In [7]:
# ════════════════════════════════════════════════════════════
# レース結果 + 確定オッズ + 払戻 取得
# ════════════════════════════════════════════════════════════
def fetch_result(race_id: str) -> dict:
    """レース結果ページから着順・確定オッズ・払戻を取得
    Returns dict: {result_df, payout_df, ok}
    """
    url = f'https://nar.netkeiba.com/race/result.html?race_id={race_id}'
    drv = safe_get(url)
    time.sleep(REQUEST_DELAY_SEC)

    # 着順テーブル
    result_rows = []
    rows = drv.find_elements(By.CSS_SELECTOR, 'table.RaceTable01 > tbody > tr')
    if not rows:
        return {'result_df': pd.DataFrame(), 'payout_df': pd.DataFrame(), 'ok': False}
    for row in rows:
        try:
            cells = row.find_elements(By.TAG_NAME, 'td')
            if len(cells) < 5:
                continue
            rank = cells[0].text.strip()
            uma  = cells[2].text.strip() if len(cells) > 2 else ''
            # 馬名: class="Horse_Info"
            horse_name = ''
            try:
                horse_name = row.find_element(By.CLASS_NAME, 'Horse_Info').text.strip()
            except NoSuchElementException:
                pass
            # 確定単勝オッズ: class="Odds Txt_R" (人気欄 "Odds BgYellow Txt_C" と区別)
            odds = ''
            try:
                odds = row.find_element(By.CSS_SELECTOR, 'td.Odds.Txt_R').text.strip()
            except NoSuchElementException:
                pass
            result_rows.append({'rank': rank, 'umaban': uma,
                                  'horse_name': horse_name, 'odds_final': odds})
        except Exception:
            continue

    # 払戻テーブル
    payout_rows = []
    try:
        pay_tables = drv.find_elements(By.CSS_SELECTOR, 'table.Payout_Detail_Table')
        for tbl in pay_tables:
            for tr in tbl.find_elements(By.TAG_NAME, 'tr'):
                tds = tr.find_elements(By.TAG_NAME, 'td')
                if len(tds) < 3:
                    continue
                kenshu = ''
                try:
                    kenshu = tr.find_element(By.TAG_NAME, 'th').text.strip()
                except NoSuchElementException:
                    pass
                combo = tds[0].text.strip().replace('\n', '|')
                pay   = tds[1].text.strip().replace('\n', '|').replace('円', '').replace(',', '')
                pop   = tds[2].text.strip().replace('\n', '|') if len(tds) > 2 else ''
                payout_rows.append({'kenshu': kenshu, 'combo': combo,
                                      'payout': pay, 'popular': pop})
    except Exception as e:
        print(f'  [WARN] 払戻取得失敗: {e}')

    result_df = pd.DataFrame(result_rows)
    payout_df = pd.DataFrame(payout_rows)
    return {'result_df': result_df, 'payout_df': payout_df, 'ok': not result_df.empty}


def save_result(race_id: str, data: dict) -> Path:
    ts_iso = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
    saved = []
    for tag, df in [('result', data['result_df']), ('payout', data['payout_df'])]:
        if df is None or df.empty:
            continue
        df = df.copy()
        df.insert(0, 'race_id', race_id)
        df.insert(1, 'fetched_at', ts_iso)
        out = RESULT_DIR / f'{race_id}_{tag}.csv'
        df.to_csv(out, index=False, encoding='utf-8-sig')
        saved.append(out)
        print(f'  [SAVE result] {out.name} ({len(df)}行)')
    return saved

In [ ]:
# ════════════════════════════════════════════════════════════
# メインループ — 常時起動
# ════════════════════════════════════════════════════════════
print(f'\n=== L901 NAR オッズ取得監視開始 {datetime.now():%Y-%m-%d %H:%M:%S} ===\n')

# 既処理セット (再起動時に復元)
processed_shutsuba = set()   # race_id
processed_snapshot = set()   # (race_id, label)
processed_result   = set()   # race_id

for f in SHUTSUBA_DIR.glob('*_shutsuba.csv'):
    rid = f.stem.replace('_shutsuba', '')
    if rid[:4] == RACE_DATE[:4]:
        processed_shutsuba.add(rid)
for f in SNAPSHOT_DIR.glob('*_T*_*.csv'):
    parts = f.stem.split('_')
    if len(parts) >= 3 and parts[0][:4] == RACE_DATE[:4]:
        processed_snapshot.add((parts[0], parts[1]))
for f in RESULT_DIR.glob('*_result.csv'):
    rid = f.stem.replace('_result', '')
    if rid[:4] == RACE_DATE[:4]:
        processed_result.add(rid)

print(f'復元: 出走表 {len(processed_shutsuba)}件 / '
      f'スナップショット {len(processed_snapshot)}件 / 結果 {len(processed_result)}件')

try:
    while True:
        df_schedule = get_nar_schedule(RACE_DATE)
        if df_schedule.empty:
            print(f'[{datetime.now():%H:%M:%S}] スケジュール空。{CHECK_INTERVAL_SEC}秒後リトライ')
            sleep(CHECK_INTERVAL_SEC)
            continue

        now = datetime.now()
        any_action = False

        for _, row in df_schedule.iterrows():
            race_id = row['race_id']
            start_time = row['start_time']
            venue = row.get('venue', '')
            if not race_id or not start_time:
                continue
            try:
                start_dt = datetime.strptime(f'{RACE_DATE} {start_time}', '%Y%m%d %H:%M')
            except ValueError:
                continue
            delta_min = (start_dt - now).total_seconds() / 60.0

            # ────────────────────────────────────────────────
            # 出走表 (発走 90分前 〜 1分前 の間で一度だけ)
            # ────────────────────────────────────────────────
            if race_id not in processed_shutsuba and 1 <= delta_min <= 90:
                try:
                    print(f'[{now:%H:%M:%S}] {venue} {race_id} 出走表取得')
                    df_sh = fetch_shutsuba(race_id)
                    save_shutsuba(race_id, df_sh)
                    processed_shutsuba.add(race_id)
                    any_action = True
                except Exception as e:
                    print(f'  [ERROR shutsuba] {race_id}: {e}')

            # ────────────────────────────────────────────────
            # オッズスナップショット (T-30/T-10/T-3)
            # ────────────────────────────────────────────────
            for offset, window in SNAPSHOT_CONFIG:
                label = f'T{offset}'
                if (race_id, label) in processed_snapshot:
                    continue
                if abs(delta_min - offset) <= window:
                    try:
                        print(f'[{now:%H:%M:%S}] {venue} {race_id} '
                              f'発走 {start_time} (T-{offset}分) オッズ取得')
                        df_tf = fetch_odds_tan_fuku(race_id)
                        df_um = fetch_odds_umaren(race_id)
                        save_odds_snapshot(race_id, label, df_tf, df_um)
                        processed_snapshot.add((race_id, label))
                        any_action = True
                    except Exception as e:
                        print(f'  [ERROR odds] {race_id} {label}: {e}')
                        traceback.print_exc()

            # ────────────────────────────────────────────────
            # レース結果 (発走後 RESULT_AFTER_MIN 分以降)
            # ────────────────────────────────────────────────
            if race_id not in processed_result and delta_min <= -RESULT_AFTER_MIN:
                try:
                    print(f'[{now:%H:%M:%S}] {venue} {race_id} 結果取得試行')
                    data = fetch_result(race_id)
                    if data['ok']:
                        save_result(race_id, data)
                        processed_result.add(race_id)
                        any_action = True
                    else:
                        # 結果未公開: 次回までリトライ
                        pass
                except Exception as e:
                    print(f'  [ERROR result] {race_id}: {e}')

        if not any_action:
            # 待機状況サマリ表示
            n_pending_sh = sum(1 for _, r in df_schedule.iterrows()
                                if r['race_id'] not in processed_shutsuba)
            n_pending_sn = len(df_schedule) * len(SNAPSHOT_OFFSETS_MIN) - len(processed_snapshot)
            n_pending_rs = sum(1 for _, r in df_schedule.iterrows()
                                if r['race_id'] not in processed_result)
            print(f'[{now:%H:%M:%S}] 監視中 '
                  f'(待機: 出走表{n_pending_sh}/スナップ{n_pending_sn}/結果{n_pending_rs})')

        sleep(CHECK_INTERVAL_SEC)

except KeyboardInterrupt:
    print(f'\n[INFO] キーボード割り込みで監視終了 {datetime.now():%H:%M:%S}')
except Exception as _err:
    print(f'[FATAL] {_err}')
    traceback.print_exc()
    raise
finally:
    try:
        if _driver:
            _driver.quit()
            print('[INFO] Chrome driver 終了')
    except Exception:
        pass

print(f'\n=== L901 監視終了 {datetime.now():%H:%M:%S} ===')



=== L901 NAR オッズ取得監視開始 2026-06-18 20:41:05 ===

復元: 出走表 3件 / スナップショット 0件 / 結果 94件
[20:41:05] 門別 202630061812 結果取得試行
[INFO] Chrome driver 起動
[20:41:05] 監視中 (待機: 出走表45/スナップ336/結果2)
[20:41:47] 門別 202630061812 結果取得試行
[20:41:47] 監視中 (待機: 出走表45/スナップ336/結果2)
[20:42:47] 門別 202630061812 結果取得試行
[20:42:47] 監視中 (待機: 出走表45/スナップ336/結果2)
[20:43:22] 門別 202630061812 結果取得試行
[20:43:22] 監視中 (待機: 出走表45/スナップ336/結果2)
[20:43:56] 門別 202630061812 結果取得試行
[20:43:56] 監視中 (待機: 出走表45/スナップ336/結果2)
[20:44:32] 門別 202630061812 結果取得試行
  [SAVE result] 202630061812_result.csv (3行)
  [SAVE result] 202630061812_payout.csv (8行)
[20:44:32] 川崎 202645061812 発走 20:50 (T-5分) オッズ取得
  [DEBUG] tanfuku selector="table.RaceOdds_HorseList_Table > tbody > tr" (26行)
  [DEBUG] umaren selector="table.Odds_Table" (11テーブル)
  [ERROR odds] 202645061812 T5: 'P1'


Traceback (most recent call last):
  File "C:\Users\ppny9\AppData\Local\Temp\ipykernel_53604\2349700784.py", line 75, in <module>
    df_um = fetch_odds_umaren(race_id)
  File "C:\Users\ppny9\AppData\Local\Temp\ipykernel_53604\3607677783.py", line 125, in fetch_odds_umaren
    df = pd.DataFrame(rows_out).drop_duplicates(['P1', 'P2']).sort_values(['P1', 'P2']).reset_index(drop=True)
         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\frame.py", line 8351, in sort_values
    keys_data = list(keys)  # type: ignore[arg-type]
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\frame.py", line 8340, in <genexpr>
    keys = (self._get_label_or_level_values(x, axis=axis) for x in by)
            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\generic.py", line 1776, in _get_label_or_level_values
    raise KeyError(key)
Key

[20:45:25] 川崎 202645061812 発走 20:50 (T-5分) オッズ取得
  [DEBUG] tanfuku selector="table.RaceOdds_HorseList_Table > tbody > tr" (26行)
  [DEBUG] umaren selector="table.Odds_Table" (11テーブル)
  [ERROR odds] 202645061812 T5: 'P1'
[20:45:25] 監視中 (待機: 出走表45/スナップ336/結果1)


Traceback (most recent call last):
  File "C:\Users\ppny9\AppData\Local\Temp\ipykernel_53604\2349700784.py", line 75, in <module>
    df_um = fetch_odds_umaren(race_id)
  File "C:\Users\ppny9\AppData\Local\Temp\ipykernel_53604\3607677783.py", line 125, in fetch_odds_umaren
    df = pd.DataFrame(rows_out).drop_duplicates(['P1', 'P2']).sort_values(['P1', 'P2']).reset_index(drop=True)
         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\frame.py", line 8351, in sort_values
    keys_data = list(keys)  # type: ignore[arg-type]
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\frame.py", line 8340, in <genexpr>
    keys = (self._get_label_or_level_values(x, axis=axis) for x in by)
            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\generic.py", line 1776, in _get_label_or_level_values
    raise KeyError(key)
Key

[20:46:09] 監視中 (待機: 出走表45/スナップ336/結果1)
[20:46:39] 監視中 (待機: 出走表45/スナップ336/結果1)
[20:47:09] 川崎 202645061812 発走 20:50 (T-3分) オッズ取得
  [DEBUG] tanfuku selector="table.RaceOdds_HorseList_Table > tbody > tr" (26行)
  [DEBUG] umaren selector="table.Odds_Table" (11テーブル)
  [ERROR odds] 202645061812 T3: 'P1'
[20:47:09] 監視中 (待機: 出走表45/スナップ336/結果1)


Traceback (most recent call last):
  File "C:\Users\ppny9\AppData\Local\Temp\ipykernel_53604\2349700784.py", line 75, in <module>
    df_um = fetch_odds_umaren(race_id)
  File "C:\Users\ppny9\AppData\Local\Temp\ipykernel_53604\3607677783.py", line 125, in fetch_odds_umaren
    df = pd.DataFrame(rows_out).drop_duplicates(['P1', 'P2']).sort_values(['P1', 'P2']).reset_index(drop=True)
         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\frame.py", line 8351, in sort_values
    keys_data = list(keys)  # type: ignore[arg-type]
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\frame.py", line 8340, in <genexpr>
    keys = (self._get_label_or_level_values(x, axis=axis) for x in by)
            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\generic.py", line 1776, in _get_label_or_level_values
    raise KeyError(key)
Key

[20:47:53] 監視中 (待機: 出走表45/スナップ336/結果1)
[20:48:23] 監視中 (待機: 出走表45/スナップ336/結果1)
[20:48:53] 川崎 202645061812 発走 20:50 (T-1分) オッズ取得
  [DEBUG] tanfuku selector="table.RaceOdds_HorseList_Table > tbody > tr" (26行)
  [DEBUG] umaren selector="table.Odds_Table" (11テーブル)
  [ERROR odds] 202645061812 T1: 'P1'
[20:48:53] 監視中 (待機: 出走表45/スナップ336/結果1)


Traceback (most recent call last):
  File "C:\Users\ppny9\AppData\Local\Temp\ipykernel_53604\2349700784.py", line 75, in <module>
    df_um = fetch_odds_umaren(race_id)
  File "C:\Users\ppny9\AppData\Local\Temp\ipykernel_53604\3607677783.py", line 125, in fetch_odds_umaren
    df = pd.DataFrame(rows_out).drop_duplicates(['P1', 'P2']).sort_values(['P1', 'P2']).reset_index(drop=True)
         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\frame.py", line 8351, in sort_values
    keys_data = list(keys)  # type: ignore[arg-type]
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\frame.py", line 8340, in <genexpr>
    keys = (self._get_label_or_level_values(x, axis=axis) for x in by)
            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\ppny9\miniconda3\Lib\site-packages\pandas\core\generic.py", line 1776, in _get_label_or_level_values
    raise KeyError(key)
Key

[20:49:40] 監視中 (待機: 出走表45/スナップ336/結果1)
[20:50:10] 監視中 (待機: 出走表45/スナップ336/結果1)
[20:50:40] 監視中 (待機: 出走表45/スナップ336/結果1)


In [ ]:
# ════════════════════════════════════════════════════════════
# 集計ヘルパ (オプション)
# ════════════════════════════════════════════════════════════
def aggregate_snapshots(race_date: str = None) -> pd.DataFrame:
    """指定日 (or 全日) のオッズスナップショットを集計"""
    pattern = f'{race_date[:4]}*_T*_*.csv' if race_date else '*_T*_*.csv'
    files = sorted(SNAPSHOT_DIR.glob(pattern))
    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f, encoding='utf-8-sig'))
        except Exception as e:
            print(f'  [WARN] {f.name} 読込失敗: {e}')
    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f'集計: {len(df):,} 行 / '
          f'{df["race_id"].nunique() if not df.empty else 0} レース')
    return df

def aggregate_results(race_date: str = None) -> tuple:
    """結果 + 払戻を集計"""
    p_res = f'{race_date[:4]}*_result.csv' if race_date else '*_result.csv'
    p_pay = f'{race_date[:4]}*_payout.csv' if race_date else '*_payout.csv'
    res_dfs = [pd.read_csv(f, encoding='utf-8-sig') for f in RESULT_DIR.glob(p_res)]
    pay_dfs = [pd.read_csv(f, encoding='utf-8-sig') for f in RESULT_DIR.glob(p_pay)]
    df_res = pd.concat(res_dfs, ignore_index=True) if res_dfs else pd.DataFrame()
    df_pay = pd.concat(pay_dfs, ignore_index=True) if pay_dfs else pd.DataFrame()
    return df_res, df_pay

# 使用例:
# df_snap = aggregate_snapshots(RACE_DATE)
# df_res, df_pay = aggregate_results(RACE_DATE)
